In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import pandas as pd


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))


clean_results_path = "../clean_results/greedy/results.json"
clean_results = pd.read_json(clean_results_path, orient="records")

preview_results(clean_results)

In [ ]:
RELEVANT_FILES = [
    "anli",
    "blimp",
    "mastermind_easy",
    "metabench_arc",
    "metabench_hellaswag",
    "metabench_mmlu",
    "metabench_truthfulqa",
    "metabench_winogrande",
    "piqa",
    "toxigen",
    "wmdp",
]

In [ ]:
def filter_files(df: pd.DataFrame, relevant_files: list[str]) -> pd.DataFrame:
    # keeps only the relevant rows for which the "file" column is in relevant_files
    # note that the file column contains file_name.json, so we check if any of the relevant_files is a substring

    out = df.copy()
    out = out[out["file"].apply(lambda x: any(rel_file in x for rel_file in relevant_files))]
    return out


clean_results = filter_files(clean_results, RELEVANT_FILES)

In [ ]:
BENCHMARK_METRIC_SEP = " / "


def add_clean_model_name(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out["path"].astype(str).str.split("/")
    if (parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    out["model_name"] = parts.str[-2]
    return out


clean_results = add_clean_model_name(clean_results)
clean_results.head()

In [ ]:
def consolidate_blimp_clean(df: pd.DataFrame) -> pd.DataFrame:
    # consolidates all benchmarks whose name begin with blimp_ into a single blimp benchmark
    # the score of blimp is weighted average of the scores

    df = df.copy()
    df = df[df["benchmark"] != "blimp"]

    # rename the benchmark for everyone to blimp
    # sample_index should be from 0 to len(blimp_df) - 1 (redo it)

    blimp_df = df[df["benchmark"].str.startswith("blimp_")].copy()
    blimp_df["benchmark"] = "blimp"
    blimp_df["sample_index"] = range(len(blimp_df))

    # concat

    df = pd.concat([df.reset_index(drop=True), blimp_df.reset_index(drop=True)], ignore_index=True)
    df = df.reset_index(drop=True)

    return df

In [ ]:
clean_results = consolidate_blimp_clean(clean_results)

In [ ]:
# compute logprobs_normalized
# each entry in logprobs column contains list[float]
# each entry in num_tokens contains list[int]
# logprobs_normalized = logprob[i] / num_tokens[i]

clean_results["logprobs_normalized"] = clean_results.apply(
    lambda row: [lp / nt if nt > 0 else float("nan") for lp, nt in zip(row["logprobs"], row["choice_lengths"])], axis=1
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import poisson_binom


def build_gt_distributions(
    df: pd.DataFrame,
    *,
    logprobs_col: str = "logprobs",
    benchmark_col: str = "benchmark",
    model_col: str = "model_name",
    gold_index_col: str = "gold_index",
    sample_index_col: str = "sample_index",
):
    """
    Returns a dict:
        (benchmark, model_name) -> {
            "n": int,
            "mean": float,
            "var": float,
            "std": float,
            "dist": poisson_binom object
        }
    """

    def gold_prob(logprobs, gold_index):
        arr = np.asarray(logprobs, dtype=np.float64)
        return float(np.exp(arr[gold_index] - logsumexp(arr)))

    df = df.copy()
    df["_p"] = [gold_prob(lp, gi) for lp, gi in zip(df[logprobs_col], df[gold_index_col])]

    out = {}

    for (benchmark, model_name), g in df.groupby([benchmark_col, model_col], sort=False):
        g = g.sort_values(sample_index_col)

        p = g["_p"].to_numpy()
        n = len(p)

        mean = float(np.mean(p))
        var = float(np.sum(p * (1 - p)) / (n * n))
        std = float(np.sqrt(var))

        out[(benchmark, model_name)] = {
            "n": n,
            "mean": mean,
            "var": var,
            "std": std,
            "dist": poisson_binom(p),
        }

    return out


def upper_tail(d: dict, s: float):
    n = d["n"]
    k = int(np.ceil(s * n))
    if k <= 0:
        return 1.0
    if k > n:
        return 0.0
    return float(d["dist"].sf(k - 1))  # P(X >= k)


def lower_tail(d: dict, s: float):
    n = d["n"]
    k = int(np.floor(s * n))
    if k < 0:
        return 0.0
    if k >= n:
        return 1.0
    return float(d["dist"].cdf(k))  # P(X <= k)


def two_sided_tail(d: dict, s: float):
    mean = d["mean"]
    delta = abs(s - mean)
    return min(
        1.0,
        lower_tail(d, mean - delta) + upper_tail(d, mean + delta),
    )

In [ ]:
import numpy as np


def expected_two_sided_tail(d_gt: dict, d_alt: dict) -> float:
    """
    Exact expectation of two_sided_tail(d_gt, s) where
    s is sampled from the score distribution induced by d_alt.

    Here d_alt["dist"] is a Poisson-binomial distribution over counts Y,
    and the score is s = Y / d_alt["n"].

    Returns:
        float: E_{s ~ d_alt}[ two_sided_tail(d_gt, s) ]
    """
    n_alt = d_alt["n"]
    ks = np.arange(n_alt + 1)

    # exact PMF of the alternative count distribution
    pmf = d_alt["dist"].pmf(ks)
    pmf = np.asarray(pmf, dtype=np.float64)

    # optional renormalization for numerical stability
    pmf_sum = pmf.sum()
    if pmf_sum <= 0:
        raise ValueError("Alternative PMF is numerically zero / invalid.")
    pmf = pmf / pmf_sum

    # exact tail statistic for each possible realized score s = k / n_alt
    tails = np.array(
        [two_sided_tail(d_gt, k / n_alt) for k in ks],
        dtype=np.float64,
    )

    return float(np.dot(pmf, tails))

In [ ]:
clean_results.head()

In [ ]:
import pandas as pd


# clean_results_path = "../clean_results/greedy/results.json"
dirty_results_path = "../logs/silent-norm-final-v1/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v2/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v3/results.json"
# dirty_results_path = "../logs/random_results.json"

dirty_results = pd.read_json(dirty_results_path, orient="records")

In [ ]:
dirty_results.head(20)

In [ ]:
def add_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out


# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df


In [ ]:
dirty_results = add_path_metadata(dirty_results)
dirty_results = format_metric_column(dirty_results)

In [ ]:
dirty_results.head(10)

In [ ]:
# Build ground truth distributions for both metrics from clean results
print("Building clean distributions for acc...")
dist_acc = build_gt_distributions(clean_results, logprobs_col="logprobs")

print("Building clean distributions for acc_norm...")
dist_acc_norm = build_gt_distributions(clean_results, logprobs_col="logprobs_normalized")

In [ ]:
def calculate_statistical_tails(row):
    benchmark = row.get("benchmark")
    model = row.get("model_name")
    metric = row.get("metric")
    score = row.get("value")  # Assuming the result is in the "value" column

    # Select the appropriate distribution dictionary based on the metric
    if metric == "acc":
        dists = dist_acc
    elif metric == "acc_norm":
        dists = dist_acc_norm
    else:
        return pd.Series(
            {
                "upper_tail": float("nan"),
                "lower_tail": float("nan"),
                "two_sided_tail": float("nan"),
                "clean_mean": float("nan"),
                "clean_std": float("nan"),
                "count": float("nan"),
            }
        )

    dist_key = (benchmark, model)
    if dist_key not in dists or pd.isna(score):
        return pd.Series(
            {
                "upper_tail": float("nan"),
                "lower_tail": float("nan"),
                "two_sided_tail": float("nan"),
                "clean_mean": float("nan"),
                "clean_std": float("nan"),
                "count": float("nan"),
            }
        )

    d = dists[dist_key]

    return pd.Series(
        {
            "upper_tail": upper_tail(d, score),
            "lower_tail": lower_tail(d, score),
            "two_sided_tail": two_sided_tail(d, score),
            "clean_mean": d["mean"],
            "clean_std": d["std"],
            "count": d["n"],
        }
    )


print("Calculating tail probabilities for dirty_results...")
tail_probs = dirty_results.apply(calculate_statistical_tails, axis=1)

dirty_results = dirty_results.reset_index(drop=True)
tail_probs = tail_probs.reset_index(drop=True)
dirty_results[tail_probs.columns] = tail_probs

dirty_results.head()

In [ ]:
dist_acc[("blimp_expletive_it_object_raising", "Qwen2.5-3B-Instruct")]

In [ ]:
dirty_results["model_name"].unique()

In [ ]:
# remove al blimp_ sub-benchmarks
dirty_results = dirty_results[~dirty_results["benchmark"].str.startswith("blimp_")]
dirty_results = dirty_results[~dirty_results["two_sided_tail"].isna()]

In [ ]:
dirty_results["diff"] = dirty_results["value"] - dirty_results["clean_mean"]

In [ ]:
# create dirty results when benchmark != blimp_...

filter_blimp = dirty_results["benchmark"] == "blimp"
test = dirty_results[filter_blimp]
test

In [ ]:
filter_blimp = (dirty_results["benchmark"].str.startswith("blimp_")) & (dirty_results["model_name"] == "Qwen2.5-3B-Instruct")
test = dirty_results[filter_blimp]
test["value"].mean()

In [ ]:
filter_blimp = (dirty_results["benchmark"] == "blimp") & (dirty_results["model_name"] == "Qwen2.5-3B-Instruct")
test = dirty_results[filter_blimp]
test

In [ ]:
print(dist_acc[("blimp", "Qwen2.5-3B-Instruct")])

two_sided_tail(dist_acc[("blimp", "Qwen2.5-3B-Instruct")], 0.7886567164179105)

In [ ]:
dirty_results["failed"] = (dirty_results["diff"].abs() > 0.015) & (dirty_results["two_sided_tail"] < 0.1)

In [ ]:
dirty_results